# 04 — Train U-Net on Kaggle (L4)

This notebook is the **Kaggle execution entry point**. Run cells top to bottom.

**Hardware:** GPU T4 × 1 (Kaggle free tier)

Steps:
1. Install packages
2. Clone / mount repo code
3. Prepare 2D slices from raw NIfTI
4. Generate patient split
5. Train U-Net
6. Evaluate on test set
7. Visualise predictions

In [ ]:
# ── 1. Install / verify packages ──────────────────────────────────────────
import subprocess, sys
def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('nibabel', 'albumentations', 'tqdm')
print('Packages ready.')

In [ ]:
# ── 2. Mount project code ─────────────────────────────────────────────────
# Option A: git clone the private repo (requires a Kaggle secret with your PAT)
# import os
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret('GITHUB_PAT')
# subprocess.run(['git', 'clone', f'https://{token}@github.com/yasmineelsayeddd/brain-tumor-segmentation.git', 'project'], check=True)
# os.chdir('project')

# Option B: repo already uploaded as Kaggle dataset (simpler)
# os.chdir('/kaggle/input/brain-tumor-seg-code')

# Option C: code cells inline (this notebook IS the entry point)
import sys
sys.path.insert(0, '/kaggle/working')  # adjust if repo is cloned elsewhere
print('sys.path[0]:', sys.path[0])

In [ ]:
# ── 3. Prepare 2D slices ──────────────────────────────────────────────────
from pathlib import Path

RAW    = Path('/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData')
OUT_2D = Path('/kaggle/working/brats2020_2d')
SPLIT  = Path('/kaggle/working/splits/default.json')
CKPT   = Path('/kaggle/working/checkpoints')

if not (OUT_2D / 'metadata.json').exists():
    print('Preparing 2D slices (this takes ~15 minutes)...')
    subprocess.run([
        sys.executable, 'scripts/prepare_data.py',
        '--input',  str(RAW),
        '--output', str(OUT_2D),
        '--keep-empty-ratio', '0.1',
    ], check=True)
else:
    print('2D slices already prepared.')

import json
with open(OUT_2D / 'metadata.json') as f:
    meta = json.load(f)
print(f'Total slices: {len(meta["slices"]):,}')

In [ ]:
# ── 4. Patient split ──────────────────────────────────────────────────────
if not SPLIT.exists():
    SPLIT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, 'scripts/make_splits.py',
        '--data-root', str(OUT_2D),
        '--output',    str(SPLIT),
    ], check=True)

with open(SPLIT) as f:
    split = json.load(f)
for s, ids in split.items():
    print(f'{s}: {len(ids)} patients')

In [ ]:
# ── 5. Train U-Net ────────────────────────────────────────────────────────
import torch
from torch.utils.data import DataLoader

from src.data.brats import BraTSDataset
from src.models.unet import UNet
from src.training.augmentation import AlbumentationsWrapper, get_train_transforms
from src.training.losses import DiceCELoss
from src.training.trainer import Trainer
from src.utils.seed import set_seed

set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))

BATCH   = 16
EPOCHS  = 50       # increase to 100 for final run
LR      = 1e-3
WORKERS = 2

train_ds = BraTSDataset(
    OUT_2D, patient_ids=split['train'],
    transform=AlbumentationsWrapper(get_train_transforms(240))
)
val_ds = BraTSDataset(OUT_2D, patient_ids=split['val'])

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=True)

print(f'Train slices: {len(train_ds):,}  |  Val slices: {len(val_ds):,}')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

model = UNet(in_channels=4, out_channels=4, base_channels=64)
print(f'Parameters: {model.parameter_count():,}')

criterion = DiceCELoss(dice_weight=0.5, ce_weight=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    num_classes=4,
    checkpoint_dir=str(CKPT),
)

history = trainer.fit(train_loader, val_loader, epochs=EPOCHS, experiment_name='unet_baseline')
print(f'\nBest val Dice: {trainer.best_val_dice:.4f}')

In [ ]:
# ── 6. Training curves ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

epochs = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs, [h['train_loss'] for h in history], label='train')
axes[0].plot(epochs, [h['val_loss']   for h in history], label='val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs, [h['train_dice'] for h in history], label='train')
axes[1].plot(epochs, [h['val_dice']   for h in history], label='val')
axes[1].set_title('Mean Dice (excl. bg)'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs, [h['lr'] for h in history], color='orange')
axes[2].set_title('Learning rate'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

In [ ]:
# ── 7. Test set evaluation ────────────────────────────────────────────────
from src.evaluation.metrics import dice_per_class, mean_dice, mean_iou

ckpt = torch.load(CKPT / 'unet_baseline_best.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
model.to(DEVICE)

test_ds = BraTSDataset(OUT_2D, patient_ids=split['test'])
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=WORKERS)

all_dice = []
all_iou  = []
with torch.no_grad():
    for images, masks in test_loader:
        logits = model(images.to(DEVICE))
        preds  = logits.argmax(dim=1)
        for p, m in zip(preds.cpu(), masks.cpu()):
            all_dice.append(dice_per_class(p, m, num_classes=4))
            all_iou.append(dice_per_class(p, m, num_classes=4))

dice_arr = np.stack(all_dice)  # (N, 3) — classes 1,2,3
class_names = ['NCR/NET', 'Edema', 'Enhancing']

print('Test set results:')
print(f'  Mean Dice: {dice_arr.mean():.4f}')
for i, name in enumerate(class_names):
    print(f'  {name:<12}: Dice={dice_arr[:,i].mean():.4f} ± {dice_arr[:,i].std():.4f}')

In [ ]:
# ── 8. Visualise predictions ──────────────────────────────────────────────
from matplotlib.colors import ListedColormap

MASK_CMAP = ListedColormap([
    (0, 0, 0, 0),
    (0.9, 0.2, 0.2, 0.6),
    (0.2, 0.8, 0.2, 0.6),
    (0.9, 0.9, 0.2, 0.6),
])

# Pick 4 slices from test set with >5% tumor
samples = []
for image, mask in test_ds:
    if (mask > 0).float().mean() > 0.05:
        samples.append((image, mask))
    if len(samples) == 4:
        break

fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4*len(samples)))
model.eval()

for row, (image, mask) in enumerate(samples):
    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(DEVICE))
    pred = logits.argmax(dim=1).squeeze(0).cpu()

    flair = image[0].numpy()
    gt_np = mask.numpy()
    pr_np = pred.numpy()
    dice  = dice_per_class(pr_np, gt_np, num_classes=4).mean()

    axes[row, 0].imshow(flair, cmap='gray'); axes[row, 0].imshow(gt_np, cmap=MASK_CMAP, vmin=0, vmax=3); axes[row, 0].set_title('FLAIR + GT');      axes[row, 0].axis('off')
    axes[row, 1].imshow(flair, cmap='gray'); axes[row, 1].imshow(pr_np, cmap=MASK_CMAP, vmin=0, vmax=3); axes[row, 1].set_title('Prediction');       axes[row, 1].axis('off')
    axes[row, 2].imshow(flair, cmap='gray'); axes[row, 2].imshow((gt_np != pr_np).astype(int), cmap='Reds', alpha=0.5); axes[row, 2].set_title(f'Error map (Dice={dice:.3f})'); axes[row, 2].axis('off')

plt.suptitle('U-Net Predictions on Test Set', fontsize=13)
plt.tight_layout()
plt.show()